# TotalSpineSeg tren Kaggle GPU – 515 anh MRI sagittal

Phan doan + danh so dot song / dia dem (L1–L5, S, T12-L1 … L5-S, ong song) cho 515 benh nhan.

**Truoc khi chay** (panel ben phai, *Settings*):
1. **Accelerator: GPU T4 x2** (khong chon P100 – PyTorch moi khong con ho tro).
2. **Internet: On** (de cai thu vien va tai trong so mo hinh ~4 GB tu GitHub).
3. *Add Input* → dataset chua `lumbar_sag_nifti.zip` (515 file `0001.nii.gz` …).

**Cach chay:**
* Chay **cell 1–4** (che do tuong tac) de kiem tra cai dat va do thoi gian tren 3 ca.
* Neu on: **Save Version → Save & Run All (Commit)** de chay ngam toi da 12 gio, khong can mo trinh duyet.
* Ket qua: `/kaggle/working/tss_sag_results.zip` (tab *Output* cua version).
* Bi ngat giua chung? Add version Output cu lam Input roi chay lai – ca da xong se duoc bo qua.


In [ ]:
# 1. Cau hinh
import os, glob, time, shutil, subprocess
from pathlib import Path

WORK = Path('/kaggle/working')
RES = WORK / 'tss_sag'                    # ket qua giu lai (step2_output, step1_levels, preview)
TMP = Path('/tmp/tss')                    # thu muc tam, KHONG luu vao output
TSS_DATA = Path('/tmp/tss_data')          # trong so mo hinh (~4 GB) - khong dua vao output
BATCH = 25                                # so ca moi lan goi totalspineseg
TIME_BUDGET_H = 11.0                      # dung nhan lo moi sau 11 gio (Kaggle cat o 12 gio)
KEEP = ['step2_output', 'step1_levels', 'preview']

# In cau truc /kaggle/input (toi da 3 cap, 5 muc moi thu muc) de chan doan
import re, zipfile
def show_tree(root='/kaggle/input', depth=3):
    for cur, dirs, files in os.walk(root, followlinks=True):
        lvl = cur[len(root):].count(os.sep)
        if lvl > depth:
            continue
        print('  ' * lvl + os.path.basename(cur) + '/', f'({len(files)} file)')
        for f in sorted(files)[:5]:
            print('  ' * (lvl + 1) + f)
show_tree()

# Tim anh dau vao. Kaggle co the: (a) giai nen zip -> 0001.nii.gz, (b) giai nen ca .gz
# -> 0001.nii, hoac (c) de nguyen file .zip. Xu ly ca 3 truong hop.
PAT = re.compile(r'^\d{4}\.nii(\.gz)?$')
found = []
for cur, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'step2_output' in cur or 'step1' in cur:
        continue
    found += [Path(cur) / f for f in files if PAT.match(f)]
if not found:
    zips = [Path(cur) / f for cur, _, fs in os.walk('/kaggle/input', followlinks=True)
            for f in fs if f.endswith('.zip')]
    assert zips, 'Khong thay anh .nii/.nii.gz hay file .zip nao - kiem tra da Add Input dataset chua'
    print('Giai nen', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('/tmp/sag_in')
    found = [p for p in Path('/tmp/sag_in').rglob('*') if PAT.match(p.name)]
IN_DIR = found[0].parent
inputs = sorted(p for p in IN_DIR.iterdir() if PAT.match(p.name))
print('\nAnh dau vao:', IN_DIR, '-', len(inputs), 'file, vd', inputs[0].name)
assert len(inputs) == 515, f'Mong doi 515 file, thay {len(inputs)}'


In [ ]:
# 2. Cai dat (~2-4 phut). kornia<0.8: ban 0.8.x bo kornia.core.Tensor ma auglab can.
!pip install -q "totalspineseg[nnunetv2]" "kornia<0.8"
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO GPU')
assert torch.cuda.is_available(), 'Chua bat GPU: Settings -> Accelerator -> GPU T4 x2'


In [ ]:
# 3. Ham chay 1 lo, ket qua gom vao RES. Bo qua ca da xong (ca trong output version cu).
def done_ids():
    found = set()
    for p in list(RES.glob('step2_output/*.nii*')) + \
             [Path(x) for x in glob.glob('/kaggle/input/**/step2_output/*.nii*', recursive=True)]:
        found.add(p.name[:4])          # so sanh theo ma BN, khong theo duoi file
    return found

def copy_previous():
    # ket qua tu version truoc (neu da Add lam Input) -> chep vao RES de zip cuoi gom du
    for k in KEEP:
        for f in glob.glob(f'/kaggle/input/**/tss_sag/{k}/*', recursive=True):
            dst = RES / k / Path(f).name
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)

def run_batch(files, log):
    if TMP.exists():
        shutil.rmtree(TMP)
    (TMP / 'in').mkdir(parents=True)
    for f in files:
        os.symlink(f, TMP / 'in' / f.name)
    t0 = time.time()
    cmd = ['totalspineseg', str(TMP / 'in'), str(TMP / 'out'), '--data-dir', str(TSS_DATA),
           '--device', 'cuda', '--quiet', '--keep-only', *KEEP]
    r = subprocess.run(cmd, capture_output=True, text=True)
    secs = time.time() - t0
    ok = r.returncode == 0
    if ok:
        for k in KEEP:
            (RES / k).mkdir(parents=True, exist_ok=True)
            for f in (TMP / 'out' / k).glob('*'):
                shutil.move(str(f), RES / k / f.name)
    else:
        print(r.stdout[-2000:], r.stderr[-3000:])
    with open(log, 'a') as fh:
        fh.write(f"{files[0].name},{files[-1].name},{len(files)},{r.returncode},{secs:.1f}\n")
    print(f"{files[0].name}..{files[-1].name}: {len(files)} ca, {secs:.0f} s "
          f"({secs/len(files):.1f} s/ca), {'OK' if ok else 'LOI ' + str(r.returncode)}")
    return ok

RES.mkdir(parents=True, exist_ok=True)
LOG = RES / 'batch_log.csv'
if not LOG.exists():
    LOG.write_text('first,last,n,exit_code,seconds\n')
copy_previous()
print('Da xong tu truoc:', len(done_ids()))


In [ ]:
# 4. Chay thu 3 ca dau (lan dau tai trong so ~4 GB nen cham hon)
todo = [f for f in inputs if f.name[:4] not in done_ids()]
run_batch(todo[:3], LOG)


In [ ]:
# 5. Chay toan bo (theo lo), dung nhan lo moi khi gan het 12 gio
t_start = time.time()
todo = [f for f in inputs if f.name[:4] not in done_ids()]
print('Can chay:', len(todo))
for i in range(0, len(todo), BATCH):
    if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
        print('Het ngan sach thoi gian - dung. Chay version moi de lam tiep.')
        break
    run_batch(todo[i:i + BATCH], LOG)
print('Tong da xong:', len(done_ids()), '/', len(inputs))


In [ ]:
# 6. Nen ket qua de tai ve (tab Output)
missing = sorted(set(f.name[:4] for f in inputs) - done_ids())
print('Thieu:', len(missing), missing[:20])
shutil.make_archive(str(WORK / 'tss_sag_results'), 'zip', root_dir=WORK, base_dir='tss_sag')
print('Da tao', WORK / 'tss_sag_results.zip',
      f"{(WORK / 'tss_sag_results.zip').stat().st_size / 2**20:.0f} MB")
